In [1]:
# Sel 1: cek isi probabilitas hold-out
import pandas as pd

kelas = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
ph = pd.read_csv("../models/prob_holdout.csv")

print("Kolom :", list(ph.columns))
print("Baris :", len(ph))

# Tebakan model = emosi dengan probabilitas tertinggi; confidence = nilai tertingginya
kolom_prob = [f"prob_{k}" for k in kelas]
ph["prediksi"] = ph[kolom_prob].idxmax(axis=1).str.replace("prob_", "", regex=False)
ph["confidence"] = ph[kolom_prob].max(axis=1)
ph["benar"] = ph["prediksi"] == ph["label_asli"]

print("\nBaris tiap label asli:")
print(ph["label_asli"].value_counts().sort_index())
print("\nAkurasi hold-out:", round(ph["benar"].mean(), 3))
print("Jumlah probabilitas per baris (harus 1.0):", ph[kolom_prob].sum(axis=1).round(3).unique())
print("\nRingkasan confidence:")
print(ph["confidence"].describe().round(3))

Kolom : ['label_asli', 'indeks', 'prob_anger', 'prob_disgust', 'prob_fear', 'prob_joy', 'prob_sadness', 'prob_surprise']
Baris : 220

Baris tiap label asli:
label_asli
anger       46
disgust     35
fear        39
joy         42
sadness     30
surprise    28
Name: count, dtype: int64

Akurasi hold-out: 0.859
Jumlah probabilitas per baris (harus 1.0): [1.]

Ringkasan confidence:
count    220.000
mean       0.901
std        0.108
min        0.508
25%        0.889
50%        0.935
75%        0.972
max        0.988
Name: confidence, dtype: float64


In [2]:
# Sel 2: cakupan dan akurasi untuk berbagai ambang
import numpy as np

baris = []
for ambang in [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]:
    dijawab = ph[ph["confidence"] >= ambang]
    baris.append({
        "ambang": ambang,
        "teks_dijawab": len(dijawab),
        "cakupan_%": round(len(dijawab) / len(ph) * 100, 1),
        "akurasi_dijawab": round(dijawab["benar"].mean(), 3) if len(dijawab) else np.nan,
        "tidak_yakin": len(ph) - len(dijawab),
    })
print(pd.DataFrame(baris).to_string(index=False))

# Akurasi pada teks yang confidence-nya rendah dan tinggi
print("\nAkurasi jika confidence < 0.80 :", round(ph[ph["confidence"] < 0.80]["benar"].mean(), 3),
      "| jumlah:", (ph["confidence"] < 0.80).sum())
print("Akurasi jika confidence >= 0.80:", round(ph[ph["confidence"] >= 0.80]["benar"].mean(), 3),
      "| jumlah:", (ph["confidence"] >= 0.80).sum())

 ambang  teks_dijawab  cakupan_%  akurasi_dijawab  tidak_yakin
   0.50           220      100.0            0.859            0
   0.60           210       95.5            0.890           10
   0.70           201       91.4            0.900           19
   0.80           193       87.7            0.902           27
   0.85           183       83.2            0.907           37
   0.90           153       69.5            0.954           67
   0.95            88       40.0            0.977          132

Akurasi jika confidence < 0.80 : 0.556 | jumlah: 27
Akurasi jika confidence >= 0.80: 0.902 | jumlah: 193


In [3]:
# Sel 3: simpan ambang dan catatan asal-usulnya
import json

ambang_final = 0.80
dijawab = ph[ph["confidence"] >= ambang_final]

konfigurasi = {
    "ambang_tidak_yakin": ambang_final,
    "urutan_label": kelas,
    "max_length": 128,
    "kolom_teks": "teks_minimal",
    "catatan": {
        "dasar_ambang": "hold-out 220 teks (10%, stratified, seed 42)",
        "cakupan_holdout_persen": round(len(dijawab) / len(ph) * 100, 1),
        "akurasi_dijawab_holdout": round(float(dijawab["benar"].mean()), 3),
        "peringatan": "Angka dihitung di data yang sama dengan pemilihan ambang; bukan ukuran performa model final.",
    },
}

with open("../models/indobert-mood/konfigurasi_aplikasi.json", "w", encoding="utf-8") as f:
    json.dump(konfigurasi, f, indent=2, ensure_ascii=False)

print(json.dumps(konfigurasi, indent=2, ensure_ascii=False))

{
  "ambang_tidak_yakin": 0.8,
  "urutan_label": [
    "anger",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise"
  ],
  "max_length": 128,
  "kolom_teks": "teks_minimal",
  "catatan": {
    "dasar_ambang": "hold-out 220 teks (10%, stratified, seed 42)",
    "cakupan_holdout_persen": 87.7,
    "akurasi_dijawab_holdout": 0.902,
    "peringatan": "Angka dihitung di data yang sama dengan pemilihan ambang; bukan ukuran performa model final."
  }
}
